# Opdracht 4 – ETL Implementatie Data Warehouse
## Robbert & Mees

In deze opdracht implementeren wij de ETL-processen van het Data Warehouse. 
We laden data vanuit het Source Data Model (SDM) naar het DWH en passen 
Slowly Changing Dimensions Type 1 en Type 2 toe.

## Inlaadstrategie

In deze opdracht maken wij gebruik van de **Incremental Data Loading** strategie.

Reden:
- we willen alleen nieuwe en gewijzigde data verwerken
- nodig voor SCD Type 2 (historie behouden)


Setup & Connecties (NOG INVULLEN)

In [ ]:
import pandas as pd
import sqlite3
from loguru import logger
from datetime import datetime

sdm_conn = sqlite3.connect('../Source Data Model/BikeToDrive_V3_OLTP.db', timeout=30)
dwh_conn = sqlite3.connect('../Data Warehouse/BikeToDrive_DWH.db', timeout=30)
sdm_cursor = sdm_conn.cursor()
dwh_cursor = dwh_conn.cursor()

logger.info("SDM en DWH connecties succesvol.")

2026-03-29 13:41:24.461 | INFO     | __main__:<module>:10 - SDM en DWH connecties succesvol.


Dim_Klant (SCD TYPE 2) ROBBERT

In [ ]:
# Wat willen we bereiken?
# We willen Dim_Klant in het DWH vullen met deze logica:
# - nieuwe klant in SDM, nog niet in DWH -> INSERT
# - bestaande klant, maar gewijzigd -> oude actuele rij afsluiten + nieuwe INSERT
# - bestaande klant, niet gewijzigd -> niets doen.

# business key = klantnr uit SDM
# surrogate key = klant_sk in DWH

# We halen eerst brondata op uit beide klanttabellen.
# Dit doen we omdat Dim_Klant niet uit één bron komt, maar uit de tabellen:
# - Accessoire_Verkoop_Klant
# - Fiets_Verkoop_Klant
# We voegen deze dus daarom ook verticaal samen.

query_accessoire_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Accessoire_Verkoop_Klant
"""

query_fiets_klant = """
SELECT
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum
FROM Fiets_Verkoop_Klant
"""

df_accessoire_klant = pd.read_sql_query(query_accessoire_klant, sdm_conn)
df_fiets_klant = pd.read_sql_query(query_fiets_klant, sdm_conn)

# We willen deze twee losse DataFrames verticaal samenvoegen.
# We maken dus één bronset voor Dim_Klant.
df_klant_source = pd.concat(
    [df_accessoire_klant, df_fiets_klant],
    ignore_index=True
)  # ignore_index=True omdat we een nette nieuwe index willen.

# We verwijderen dubbele rijen, want we willen niet dezelfde klant twee keer in onze bronset hebben
# voordat we hem vergelijken met het DWH.
df_klant_source = df_klant_source.drop_duplicates().reset_index(drop=True)

print(df_klant_source.head())
print(f"Aantal unieke klanten in source: {len(df_klant_source)}")

# Nu halen we de actuele klantrecords op uit het DWH.
# Bij SCD Type 2 willen we namelijk alleen vergelijken met de actuele rij van een klant,
# niet met de oude historische versies.
# We herkennen de actuele regel aan: eindtijd IS NULL.

query_dwh_klant = """
SELECT
    klant_sk,
    klantnr,
    naam,
    adres,
    woonplaats,
    geslacht,
    geboortedatum,
    begintijd,
    eindtijd
FROM Dim_Klant
WHERE eindtijd IS NULL
"""

df_klant_dwh = pd.read_sql_query(query_dwh_klant, dwh_conn)

print(df_klant_dwh.head())
print(f"Aantal actuele klanten in DWH: {len(df_klant_dwh)}")

# Nu gaan we bepalen of een bestaande klant veranderd is.
# We vergelijken alleen inhoudelijke klantgegevens.
# We vergelijken dus NIET op:
# - klant_sk
# - begintijd
# - eindtijd
# want die horen bij de DWH-historie en niet bij de business-inhoud van de klant.

# We maken een functie:
# - source_row -> één klant uit de bron
# - dwh_row -> de actuele versie van diezelfde klant uit het DWH
# Daarna checken we: is één van de inhoudelijke velden anders?
# - True -> klant is veranderd
# - False -> klant is niet veranderd

def klant_is_changed(source_row, dwh_row):
    return (
        str(source_row["naam"]) != str(dwh_row["naam"]) or
        str(source_row["adres"]) != str(dwh_row["adres"]) or
        str(source_row["woonplaats"]) != str(dwh_row["woonplaats"]) or
        str(source_row["geslacht"]) != str(dwh_row["geslacht"]) or
        str(source_row["geboortedatum"]) != str(dwh_row["geboortedatum"])
    )

# We gaan nu bepalen of de klant nieuw is of al bestaat in het DWH.
# We doen dit omdat elke bronrij straks in één van deze categorieën valt:
# - nieuw -> nog niet in DWH
# - bestaand -> gewijzigd of ongewijzigd

new_count = 0
changed_count = 0
unchanged_count = 0

for _, src_row in df_klant_source.iterrows():  # _ is de index, die gebruiken we hier niet. src_row is één klantrecord.
    klantnr = src_row["klantnr"]  # We pakken de business key, omdat dat de sleutel uit het SDM is waarmee we dezelfde klant kunnen herkennen.

    current_match = df_klant_dwh[df_klant_dwh["klantnr"] == klantnr]  # Zit er in de actuele DWH-klanten al een rij met deze business key?

    # 1. Nieuwe klant
    if current_match.empty:
        new_count += 1

    # 2. Bestaande klant
    else:
        dwh_row = current_match.iloc[0]

        if klant_is_changed(src_row, dwh_row):
            changed_count += 1
        else:
            unchanged_count += 1

print(f"Aantal nieuwe klanten: {new_count}")
print(f"Aantal gewijzigde klanten: {changed_count}")
print(f"Aantal ongewijzigde klanten: {unchanged_count}")

# We gaan nu de echte ETL uitvoeren voor Dim_Klant.
# Hierbij verwerken we:
# - nieuwe klanten -> INSERT
# - gewijzigde klanten -> oude actuele rij afsluiten + nieuwe INSERT
# - ongewijzigde klanten -> niets doen

# We maken één timestamp voor deze laadrun.
# Deze gebruiken we:
# - als begintijd voor nieuwe actuele rijen
# - als eindtijd voor oude rijen die niet meer actueel zijn
now_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# Deze tellers gebruiken we om straks te zien wat de ETL echt gedaan heeft.
inserted_count = 0
updated_count = 0
unchanged_count = 0

logger.info("Start ETL voor Dim_Klant (SCD Type 2)")

for _, src_row in df_klant_source.iterrows():
    klantnr = src_row["klantnr"]

    # Zoek op business key of deze klant al als actuele rij in het DWH staat.
    current_match = df_klant_dwh[df_klant_dwh["klantnr"] == klantnr]

    # 1. Nieuwe klant -> INSERT
    if current_match.empty:
        # De klant bestaat nog niet in het DWH.
        # We voegen daarom een nieuwe actuele rij toe.
        # eindtijd = NULL, want deze rij is actueel.
        dwh_conn.execute("""
            INSERT INTO Dim_Klant (
                klantnr,
                naam,
                adres,
                woonplaats,
                geslacht,
                geboortedatum,
                begintijd,
                eindtijd
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, NULL)
        """, (
            int(src_row["klantnr"]),
            src_row["naam"],
            src_row["adres"],
            src_row["woonplaats"],
            src_row["geslacht"],
            src_row["geboortedatum"],
            now_ts
        ))

        inserted_count += 1

    # 2. Bestaande klant -> check of gewijzigd
    else:
        # current_match is nog een DataFrame.
        # Met iloc[0] pakken we de eerste (en actuele) rij.
        dwh_row = current_match.iloc[0]

        # We checken of de inhoudelijke klantgegevens veranderd zijn.
        if klant_is_changed(src_row, dwh_row):

            # De klant is gewijzigd.
            # Bij SCD Type 2 overschrijven we de oude rij NIET.
            # In plaats daarvan:
            # 1. sluiten we de oude actuele rij af met een eindtijd
            # 2. voegen we een nieuwe actuele rij toe

            # Stap 1: oude actuele rij afsluiten
            dwh_conn.execute("""
                UPDATE Dim_Klant
                SET eindtijd = ?
                WHERE klant_sk = ?
            """, (
                now_ts,
                int(dwh_row["klant_sk"])
            ))

            # Stap 2: nieuwe actuele versie toevoegen
            dwh_conn.execute("""
                INSERT INTO Dim_Klant (
                    klantnr,
                    naam,
                    adres,
                    woonplaats,
                    geslacht,
                    geboortedatum,
                    begintijd,
                    eindtijd
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, NULL)
            """, (
                int(src_row["klantnr"]),
                src_row["naam"],
                src_row["adres"],
                src_row["woonplaats"],
                src_row["geslacht"],
                src_row["geboortedatum"],
                now_ts
            ))

            updated_count += 1

        else:
            # De klant bestaat al en is niet veranderd.
            # We hoeven dus niets te doen.
            unchanged_count += 1

# We slaan alle wijzigingen definitief op in het DWH.
dwh_conn.commit()

logger.info(
    f"Dim_Klant klaar | inserted = {inserted_count}, "
    f"updated_scd2 = {updated_count}, unchanged = {unchanged_count}"
)

# Tot slot controleren we het resultaat.
# We sorteren op klantnr en klant_sk zodat historische versies netjes onder elkaar staan.
result_df = pd.read_sql_query("""
    SELECT *
    FROM Dim_Klant
    ORDER BY klantnr, klant_sk
""", dwh_conn)

print(result_df)

   klantnr            naam           adres woonplaats geslacht geboortedatum
0        1      Jan Jansen   Kerkstraat 12  Amsterdam        M    1985-03-22
1        2  Sophie de Boer     Lindelaan 8  Rotterdam        V    1990-07-11
2        3   Pieter Visser   Havenstraat 3   Den Haag        M    1978-11-05
3        4       Emma Smit    Boomgaard 22    Haarlem        V    1995-02-18
4        5      Tom Bakker  Stationsweg 44     Leiden        M    1982-09-09
Aantal unieke klanten in source: 30
Empty DataFrame
Columns: [klant_sk, klantnr, naam, adres, woonplaats, geslacht, geboortedatum, begintijd, eindtijd]
Index: []
Aantal actuele klanten in DWH: 0


Dim_Accessoire (SCD TYPE 1) ROBBERT

Dim_Datum ROBBERT

Fct_Verkoop ROBBERT


Dim_Fiets (SCD Type 2) MEES

Dim_Monteur (SCD Type 1) MEES

Fct_Onderhoud MEES

Fct_Inkoop MEES